In [2]:
# 1. Confirm we have a GPU with enough VRAM for 7B in bf16 (~15 GB weights).
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 4060 Laptop GPU, 8188 MiB


In [3]:
import os
if not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/afsp-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD 

/home/prnamhr/projects/Style-Aware-MT/notebooks/Style-Aware-MT
843fe38


In [4]:
!pip install -q "transformers==5.12.1" "accelerate==1.14.0" "PyYAML==6.0.3"

In [7]:
!python -m src.infer.run --condition zeroshot --config configs/qwen_smoke.yaml

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 4 files:   0%|                                  | 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|       |  0.00B / 3.95GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 15.2GB            
Reconstructing (incomplete total...):   0%|       |  0.00B / 15.2GB            
Reconstructing (incomplete total...):   8%|▌      | 1.15GB / 15.2GB, 2.16MB/s  
Reconstructing (incomplete total...):  44%|███    | 6.63GB / 15.2GB, 4.38MB/s  
Reconstructing (incomplete total...):  75%|█████▏ | 11.4GB / 15.2GB, 5.43MB/s  
Reconstructing (incomplete total...): 100%|███████| 15.2GB / 15.2GB, 4.60MB/s  

Fetching 4 files: 100%|█████████████████████████| 4/4 [57:36<00:00, 864.19s/it]
Download complete: : ██████████████████████████████████████| 13.1GB, 4.07MB/s  
Download complete: : ██████████████████████████████████████| 13.1GB, 4.07MB/s  
Reconstruction complete: 100%|████████

In [8]:
# 5. Verify outputs: 5 predictions + non-zero token accounting.
!cat outputs/zeroshot_val_usage.json
print('--- predictions ---')
!wc -l outputs/zeroshot_val.jsonl
import json
with open('outputs/zeroshot_val.jsonl', encoding='utf-8') as f:
    row = json.loads(f.readline())
print('sample prediction:', row['prediction'][:300])

{
  "condition": "zeroshot",
  "model": "Qwen/Qwen2.5-7B-Instruct",
  "calls": 5,
  "prompt_tokens": 1282,
  "completion_tokens": 178,
  "cost_usd": 0.0
}

--- predictions ---
5 outputs/zeroshot_val.jsonl
sample prediction: Blessed are the righteous who drink from these rivers, O Thou, the Almighty, the Forgiver, unto whom approacheth none save those who draw nigh by power and might.


## Retrieval / few-shot smoke 

In [9]:
!pip install -q "sentence-transformers==5.5.1"

In [10]:
!python -m src.retrieval.build_index --config configs/qwen_smoke.yaml

Embedding 10860 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Batches: 100%|███████████████████████████████| 340/340 [00:13<00:00, 25.58it/s]
Wrote index to data/knn_index/ : embeddings (10860, 1024), 10860 pairs


In [11]:
# Target-register centroid over train targets -> results/stylometrics_centroid.json.
!python -m src.eval.stylometrics --build-centroid


Target-register centroid  (n=10860)  -> results/stylometrics_centroid.json
---------------------------------------
  lex_density  mean 0.4344   std 0.1101
  ttr          mean 0.8540   std 0.1085
  root_ttr     mean 4.0437   std 1.0426
  marker_rate  mean 0.0327   std 0.0567



In [12]:
# knn_fewshot: cosine top-k retrieval (isolates relevance-based selection over "having examples").
!python -m src.infer.run --condition knn_fewshot --config configs/qwen_smoke.yaml

Retrieving k=4 exemplars for 5 sources (most_similar_last) ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 157.79it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
Generating 5 translations with Qwen/Qwen2.5-7B-Instruct (knn_fewshot) ...
  5/5
Wrote outputs/knn_fewshot_val.jsonl
Usage: {'calls': 5, 'prompt_tokens': 5696, 'completion_tokens': 189, 'cost_usd': 0.0}


In [13]:
!python -m src.infer.run --condition afsp_full --config configs/qwen_smoke.yaml

afsp_full: selecting k=4 for 5 sources (most_similar_last) ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100%|██████████████████████| 339/339 [00:01<00:00, 227.53it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.
Generating 5 translations with Qwen/Qwen2.5-7B-Instruct (afsp_full) ...
  5/5
Wrote outputs/afsp_full_val.jsonl
Usage: {'calls': 5, 'prompt_tokens': 5458, 'completion_tokens': 197, 'cost_usd': 0.0}


In [5]:
!python manage.py afsp_sweep --config configs/qwen_smoke.yaml --ks 8 --lambdas 0.0 1.0

Loading weights: 100%|██████████████████████| 339/339 [00:03<00:00, 106.09it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
[afsp_k8_l0] selecting k=8 (lambda=0.0) for 5 sources ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 1016.45it/s]
[afsp_k8_l0] generating 5 translations with Qwen/Qwen2.5-7B-Instruct ...
[afsp_k8_l1] selecting k=8 (lambda=1.0) for 5 sources ...
[afsp_k8_l1] generating 5 translations with Qwen/Qwen2.5-7B-Instruct ...
Loading weights: 100%|██████████████████████| 339/339 [00:00<00:00, 960.77it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
[afsp_zeroshot] generating 5 zero-shot translations with Qwen/Qwen2.5-7B-Instruct ...

AFSP sweep (val, 3 cells)  -- register fidelity first
tag            k  lambda  n  chrF   BLEU   marker_rate  stylo_dist  register_fit  <-
--------------------------------

In [14]:
# Verify the few-shot path beyond "no crash": 5 rows each, clean English, no `Source:` leakage.
import json, os
for cond in ("knn_fewshot", "afsp_full"):
    path = f"outputs/{cond}_val.jsonl"
    if not os.path.exists(path):
        print(f"=== {cond}: MISSING ({path}) -- its generation cell did not complete; "
              f"re-run it and read its traceback ===\n")
        continue
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    r = rows[0]
    pred = r["prediction"]
    print(f"=== {cond}: {len(rows)} rows, {sum('error' in x for x in rows)} errors ===")
    print("sample prediction[:250]:", pred[:250])
    print("`Source:` leakage in prediction:", "Source:" in pred)
    print()

=== knn_fewshot: 5 rows, 0 errors ===
sample prediction[:250]: Jewels of the mysteries in the ascent of journeys for him who desireth to draw nigh to God, the Almighty, the Forgiver; blessed indeed are the righteous who drink of these rivers.
`Source:` leakage in prediction: False

=== afsp_full: 5 rows, 0 errors ===
sample prediction[:250]: O God, the jewels of mysteries in the stages of journeys are for him who desireth to draw nigh unto Thee, the All-Merciful, the All-Powerful. Verily, blessed are the righteous who drink from these rivers.
`Source:` leakage in prediction: False

